# Swift and Fermi

## Swift

<img src="images/swift.png" alt="Swift and Fermi mission comparison" width="600"/>


<img src="images/swift_1.png" alt="Swift and Fermi mission comparison" width="600"/>


<img src="images/xrt.png" alt="Swift and Fermi mission comparison" width="600"/>


<img src="images/uvot.png" alt="Swift and Fermi mission comparison" width="600"/>


Check https://gcn.gsfc.nasa.gov/swift.html for further details for the content of each notice

# Fermi


<img src="images/fermi.png" alt="Swift and Fermi mission comparison" width="600"/> 




<img src="images/fermi_1.png" alt="Swift and Fermi mission comparison" width="600"/>

<img src="images/fermi_2.png" alt="Swift and Fermi mission comparison" width="600"/>

Check https://gcn.gsfc.nasa.gov/fermi.html for further details for the content of each notice

## Get familiar with notice content

https://gcn.gsfc.nasa.gov/fermi_grbs.html

https://gcn.gsfc.nasa.gov/swift_grbs.html


## Let's try our first kafka connection

set 

```python
"auto.offset.reset": "earliest"
```

to receive all the messages you missed since the last time you had the listener active

set 

```python
"auto.offset.reset": "latest" 
```

to receive ONLY NEW  messages you missed since the last time you had the listener active


In [ ]:
!pip install gcn-kafka xmltodict

from gcn_kafka import Consumer
from confluent_kafka import TopicPartition
import os
import json
import datetime 
import xmltodict
from astropy.coordinates import SkyCoord

# Connect as a consumer (client "Mac MOC")
# Warning: don't share the client secret with others.
CONFIG = {"group.id": "", "auto.offset.reset": "earliest"}
consumer = Consumer(config=CONFIG,
                    client_id='##',
                    client_secret='##',
                    domain='gcn.nasa.gov')


In [ ]:
import email

def write_json(text_data, output_dir="./gcn_alerts"):
    """Parse GCN text format and extract relevant information, then save to JSON."""
    
    # Convert bytes to string if necessary
    if isinstance(text_data, bytes):
        text_data = text_data.decode('utf-8')
    
    # Parse using email module
    msg = email.message_from_string(text_data)
    result = dict(msg)
    
    try:
        # Create output directory if it doesn't exist
        os.makedirs(output_dir, exist_ok=True)
        
        # Create filename using trigger number or timestamp
        if 'TRIGGER_NUM' in result:
            filename = f"GRB_{result['TRIGGER_NUM']}.json"
        else:
            timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
            filename = f"GRB_{timestamp}.json"
        
        filepath = os.path.join(output_dir, filename)
        
        with open(filepath, 'w') as f:
            json.dump(result, f, indent=2)
        
        print(f"Saved to: {filepath}")
    
    except Exception as e:
        print("Error processing GCN text notice:", e)
    
    return result


In [ ]:
# Time ranges are specified as Kafka times (milliseconds since the unix epoch)

t1 = 1

t2 = 1

timestamp1 = int((datetime.datetime.now() - datetime.timedelta(days=t1)).timestamp() * 1000)
timestamp2 = timestamp1 + t2*86400000 # +7 days

In [ ]:


print(f"Fetching messages between {timestamp1} and {timestamp2}")

# Subscribe to topics and receive alerts
topics = 'gcn.classic.text.FERMI_GBM_ALERT'
# topics = 'gcn.classic.text.SWIFT_BAT_GRB_POS_ACK'
# topics = 'gcn.classic.text.SWIFT_XRT_POSITION'
# topics = 'gcn.classic.text.SWIFT_UVOT_POS'

start = consumer.offsets_for_times(
    [TopicPartition(topics, 0, timestamp1)])
end = consumer.offsets_for_times(
    [TopicPartition(topics, 0, timestamp2)])


consumer.assign(start)

# Calculate the number of messages to consume, ensuring it's within valid range
num_messages = end[0].offset - start[0].offset


for message in consumer.consume(abs(num_messages), timeout=1):
    
# while True:
#     for message in consumer.consume(timeout=1):
        if message.error():
            print(message.error())
            continue
        # Print the topic and message ID
        print(f'topic={message.topic()}, offset={message.offset()}')
        value = message.value()
        # xml_dict = xmltodict.parse(value)
        write_json(value)